In [8]:
#simulare con Monte Carlo (Metropolis) una particella soggetta ad un Potenziale Double Well 
#in D dimensioni a contatto termico con un reservoir a temperatura T.
import jax 
import jax.numpy as jnp
import matplotlib.pyplot as plt

In [9]:
#definiamo il potenziale in modo che x1 sia la variabile lenta e x2,...,xD siano le variabili veloci
@jax.jit
def V(x, a=1.0, b=1.0):
    return a * (x[0]**2 - b)**2 + 0.5 * jnp.sum(x[1:]**2)

In [10]:
#genera una condigurazione iniziale casuale usando una gaussiana standard
@jax.jit
def generate_config(key, D):
    key, subkey = jax.random.split(key)
    x = jax.random.normal(subkey, shape=(D,))
    return x, key

In [11]:
#genera un numero casuale con una gaussiana standard
@jax.jit
def generate_normal(key):
    key, subkey = jax.random.split(key)
    x = jax.random.normal(subkey)
    return x, key

#da uniforme:
@jax.jit
def generate_uniform(key):
    key, subkey = jax.random.split(key)
    x = jax.random.uniform(subkey)
    return x, key

In [12]:
#funzione di accettazione di Metropolis
@jax.jit
def metropolis_acceptance(delta_E, T):
    A = jnp.where(delta_E <= 0, 1.0, jnp.exp(-delta_E / T))
    return A

#funzione per eseguire un passo di Metropolis
@jax.jit
def metropolis_step(x, key, T, step_size=0.1):
    # nuova configurazione proposta
    eta, key = generate_normal(key)
    x_proposed = x + step_size * eta
    # calcola la differenza di energia
    delta_E = V(x_proposed) - V(x)
    # accetta o rifiuta la nuova configurazione
    accept_prob = metropolis_acceptance(delta_E, T)
    u, key = generate_uniform(key)
    accept = u < accept_prob
    x_new = jnp.where(accept, x_proposed, x)
    return x_new, key, accept

In [13]:
#funzione per eseguire la simulazione
@jax.jit
def run_simulation(key, D, T, n_steps, step_size=0.1):
    #config. iniziale
    x, key = generate_config(key, D)
    trajectory = jnp.zeros((n_steps, D))
    acceptances = 0
    for i in range(n_steps):
        x, key, accepted = metropolis_step(x,key,T,step_size)
        trajectory = trajectory.at[i].set(x)
        acceptances += accepted
    acceptance_rate = acceptances / n_steps
    return trajectory, acceptance_rate

In [14]:
#parametri della simulazione
D = 2  # dimensioni
T = 1.0  # temperatura
n_steps = 10000  # numero di passi
step_size = 0.5  # dimensione del passo

#la simulazione
key = jax.random.PRNGKey(0)
trajectory, acceptance_rate = run_simulation(key, D, T, n_steps, step_size)
print(f"Acceptance rate: {acceptance_rate:.2f}")
#visualizzazione della traiettoria
plt.figure(figsize=(8, 6))
plt.plot(trajectory[:, 0], trajectory[:, 1], alpha=0.5)
plt.title("Traiettoria della particella nel potenziale Double Well")
plt.xlabel("x1")
plt.ylabel("x2")
plt.grid()
plt.show()


KeyboardInterrupt: 